# Session 6 — Real case: Electronics replenishment exceptions

**Salih Tutun, PhD** · DAT 5668 · Module 3 · Explain → practice together

## How to use this notebook

1. Read the short **Topic** cells (Part I). Do not scroll into practice yet.  
2. In Part II, every stop is: **explain markdown → small code cell → run → talk**.  
3. Cues:

| Cue | Meaning |
| --- | --- |
| **RUN WITH ME** | Code is ready. Read comments. Run. Do not edit. |
| **TRY THIS** | You type `<<< FILL`. Re-run. Check. |
| **CHANGE ME** | Change the marked value. Re-run. **PAIR 20s** discuss. |
| **CHECK** | Print PASS / NOT YET |

Partner rule: both can say who **releases the daily order** and who **stamps the promo override**.

![Two families at one desk](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-two-families.png)
![Hybrid design](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/hybrid.png)

*Images load from GitHub. If blank: hard-refresh or reopen.*


## Practice together tonight (change → see → discuss)

For **CHANGE ME** cells:
1. Change only the marked line  
2. Re-run that cell  
3. **PAIR 20s** — what changed?  
4. Two voices share — then we continue  

We are not hunting a secret answer. We are learning what the real desk does when YOU change a rule.


# Part I — Topic (explain before coding)

## The real desk

You own **Electronics** replenishment. Real log: `inventory_dataset_full.csv` (730 Electronics days).

| Column | Meaning |
| --- | --- |
| `Current_Inventory` | Stock you can measure (state) |
| `Demand`, `Stockout_Units` | What customers wanted / missed |
| `Action_Taken` | `Order` / `Maintain` / `Reduce` |
| `Profit`, `Holding_Cost`, `Stockout_Cost` | Money (reward language) |
| `Is_Promotional` | Campaign day — needs a document path |

![Same inventory loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/session05-loop.png)

```text
observe inventory → decide action → demand → profit/costs → next inventory
```

**Tonight:** who owns the **daily numeric order** (RL) vs **promo briefs** (LLM + human stamp)?

![Numeric path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)
![Document path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-llm-agent.png)
![Human stamp](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-handoff.png)

| Situation | Lean toward |
| --- | --- |
| Daily inventory → order with profit/costs | **RL** |
| Promo/supplier documents | **LLM** (or person) |
| This real log (both) | **Hybrid** |
| Reward unsigned / unsafe to explore | **Do not automate** |

**Teach aloud:** prediction ≠ agent. Proposal ≠ release. Stock number ≠ promo brief.

**JUMP BACK** — do not open Stop 0 until I say OPEN.


---
# Part II — Practice (small cells: explain → run)

We split code on purpose: one idea per cell so every line is readable.


## Stop 0 — Find the dataset

**Goal:** attach Drive (Colab) or confirm local files exist.

**What the next cell does**
1. Import `Path` (clean file paths)  
2. Name the Colab Drive folder  
3. Try to mount Google Drive  
4. List CSV files and check for `inventory_dataset_full.csv`


In [ ]:
# STOP 0 — RUN WITH ME
# Find inventory_dataset_full.csv (Drive in Colab, or local).

from pathlib import Path  # build file paths without messy string joins

# Where the shared course data should live in Google Colab
DRIVE_ROOT = Path("/content/drive/MyDrive/DAT5668/data")

try:
    # Only works inside Google Colab
    from google.colab import drive  # type: ignore

    # Ask permission once and attach your Google Drive
    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted.")
except ImportError:
    # Running on a laptop / Cursor — that is OK
    print("Not Colab — OK. We will look for local course data.")

# If the Drive folder exists, show which CSVs students can see
if DRIVE_ROOT.exists():
    files = sorted(p.name for p in DRIVE_ROOT.glob("*.csv"))  # all CSV names
    print("CSVs:", files)
    # Explicit yes/no for the Session 6 file
    print(
        "inventory_dataset_full.csv →",
        "YES" if "inventory_dataset_full.csv" in files else "MISSING",
    )
else:
    # Missing Drive folder is fine locally; in Colab it means remount/share
    print("Drive folder not found yet (OK if local):", DRIVE_ROOT)


## Stop 1 — Load the real Electronics desk

![Numeric desk](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-mdp.png)

We load **only** `Product == "Electronics"` rows, then print real facts.


### 1a — Imports (RUN WITH ME)

**What this cell does:** bring in tools we need for files, counting, CSV, averages, and charts.


In [ ]:
# STOP 1a-1 — RUN WITH ME · imports

from pathlib import Path          # file paths
from collections import Counter   # count actions / demand bins
import csv                        # read CSV rows as dictionaries
import statistics                 # mean() for demand, inventory, profit

# Charts are optional (some environments lack matplotlib)
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None  # no charts; numbers still work

print("Imports ready. plt =", "yes" if plt is not None else "no")


### 1a — Settings (RUN WITH ME)

**What this cell does:** name the Drive folder and the product line we study tonight.


In [ ]:
# STOP 1a-2 — RUN WITH ME · settings

DRIVE_DATA_FOLDER = "/content/drive/MyDrive/DAT5668/data"  # Colab shared data
PRODUCT = "Electronics"  # one product line from the real log

print("Looking for product:", PRODUCT)
print("Drive data folder:", DRIVE_DATA_FOLDER)


### 1a — Helper: `course_data` (RUN WITH ME)

**What this cell does:** define a function that finds a CSV on Drive first, then in local course folders.


In [ ]:
# STOP 1a-3 — RUN WITH ME · define course_data

def course_data(name: str) -> Path:
    # Build an ordered list of places to look
    candidates = []

    # 1) Prefer the Colab Drive folder when it exists
    if DRIVE_DATA_FOLDER:
        candidates.append(Path(DRIVE_DATA_FOLDER) / name)

    # 2) Then try common local / GitHub / Canvas unzip layouts
    for base in (
        Path("data"),
        Path("../../Drive-Folder-for-course/DAT5668/data"),
        Path("../Drive-Folder-for-course/DAT5668/data"),
        Path("../../../Drive-Folder-for-course/DAT5668/data"),
        Path("../../assets/data"),
        Path("assets/data"),
        Path("../../Contents/Module 4"),
        Path("../Contents/Module 4"),
        Path("/content/data"),
        Path("../data"),
    ):
        candidates.append(base / name)

    # 3) Return the first path that actually exists
    for path in candidates:
        if path.exists():
            print("Using:", path)  # so everyone sees which file loaded
            return path

    # 4) If nothing worked, stop with a clear error (path issue, not MDP issue)
    raise FileNotFoundError(
        "Could not find " + name + " — put it in MyDrive/DAT5668/data/ or Canvas Files"
    )


print("Defined course_data(name) → Path")


### 1a — Helper: `_flag` (RUN WITH ME)

**What this cell does:** turn CSV flags like `"1"` / `"True"` into real Python `True`/`False`.


In [ ]:
# STOP 1a-4 — RUN WITH ME · define _flag

def _flag(val) -> bool:
    # CSV often stores booleans as strings
    return str(val).strip() in {"1", "True", "true", "YES", "Yes"}


# Tiny demo so you see what it does
print(_flag("1"), _flag("0"), _flag("True"), _flag("false"))


### 1a — Helper: `load_electronics_desk` (RUN WITH ME)

**What this cell does:** open the CSV, keep Electronics rows, convert types, sort by date.


In [ ]:
# STOP 1a-5 — RUN WITH ME · define load_electronics_desk

def load_electronics_desk():
    rows = []  # we will append one dict per Electronics day

    # Open the CSV as text (UTF-8)
    with course_data("inventory_dataset_full.csv").open(encoding="utf-8") as handle:
        # DictReader → each row is {column_name: value}
        for row in csv.DictReader(handle):
            # Keep only tonight's product line
            if (row.get("Product") or "").strip() != PRODUCT:
                continue

            # Store a clean typed record for class
            rows.append(
                {
                    "Date": row["Date"],
                    "Current_Inventory": int(float(row["Current_Inventory"])),
                    "Demand": int(float(row["Demand"])),
                    "Units_Sold": int(float(row["Units_Sold"])),
                    "Stockout_Units": int(float(row["Stockout_Units"])),
                    "Lead_Time": int(float(row["Lead_Time"])),
                    "Price": float(row["Price"]),
                    "Is_Promotional": _flag(row.get("Is_Promotional", 0)),
                    "Is_Weekend": _flag(row.get("Is_Weekend", 0)),
                    "Action_Taken": (row.get("Action_Taken") or "").strip(),
                    "Order_Quantity": int(float(row.get("Order_Quantity") or 0)),
                    "Order_Placed": _flag(row.get("Order_Placed", 0)),
                    "Holding_Cost": float(row.get("Holding_Cost") or 0),
                    "Stockout_Cost": float(row.get("Stockout_Cost") or 0),
                    "Profit": float(row.get("Profit") or 0),
                    "Revenue": float(row.get("Revenue") or 0),
                    "Season": (row.get("Season") or "").strip(),
                    "Reorder_Point": int(float(row.get("Reorder_Point") or 0)),
                }
            )

    # Safety: empty filter means wrong file / wrong product name
    if not rows:
        raise ValueError("No Electronics rows found — check the CSV.")

    # Chronological order helps when we pick real days later
    rows.sort(key=lambda r: r["Date"])
    return rows


print("Defined load_electronics_desk() → list of day dicts")


### 1a — Run the load (RUN WITH ME)

**What this cell does:** call the loader once and keep `ROWS` for the rest of class.


In [ ]:
# STOP 1a-6 — RUN WITH ME · load once

ROWS = load_electronics_desk()  # list of Electronics day dictionaries
N_DAYS = len(ROWS)              # how many days in the log

print("Loaded Electronics days:", N_DAYS)
print("First date:", ROWS[0]["Date"], "| Last date:", ROWS[-1]["Date"])


### 1a — Real summary numbers (RUN WITH ME)

**What this cell does:** compute averages and counts students will quote in the memo.


In [ ]:
# STOP 1a-7 — RUN WITH ME · summary stats

# Average demand and inventory across all Electronics days
AVG_DEMAND = statistics.mean(r["Demand"] for r in ROWS)
AVG_INV = statistics.mean(r["Current_Inventory"] for r in ROWS)

# Subsets we will use for promo / stockout discussion
PROMO_DAYS = [r for r in ROWS if r["Is_Promotional"]]
STOCKOUT_DAYS = [r for r in ROWS if r["Stockout_Units"] > 0]

# What actions appear in the historical log?
ACTIONS = Counter(r["Action_Taken"] for r in ROWS)

print("=== REAL CASE: Electronics replenishment desk ===")
print("Days in log:", N_DAYS)
print("Avg inventory:", round(AVG_INV, 1), "| Avg demand:", round(AVG_DEMAND, 1))
print("Promo days:", len(PROMO_DAYS), f"({100 * len(PROMO_DAYS) / N_DAYS:.1f}%)")
print("Stockout days:", len(STOCKOUT_DAYS), f"({100 * len(STOCKOUT_DAYS) / N_DAYS:.1f}%)")
print("Actions in data:", dict(ACTIONS))
print("Mean profit/day: $", round(statistics.mean(r["Profit"] for r in ROWS), 1))
print("Mean holding cost/day: $", round(statistics.mean(r["Holding_Cost"] for r in ROWS), 1))
print("Mean stockout cost/day: $", round(statistics.mean(r["Stockout_Cost"] for r in ROWS), 1))
print("Reorder point in data:", ROWS[0]["Reorder_Point"])
print(
    "Sample row:",
    {
        k: ROWS[10][k]
        for k in [
            "Date",
            "Current_Inventory",
            "Demand",
            "Stockout_Units",
            "Is_Promotional",
            "Action_Taken",
            "Profit",
        ]
    },
)


### 1a — Demand bins + charts (RUN WITH ME)

**What this cell does:** bin real daily demand into $d \in \{0,1,2\}$ (readable MDP style) and plot.


In [ ]:
# STOP 1a-8 — RUN WITH ME · demand bins + optional charts

# Sort all demand values so we can cut into thirds
demands = sorted(r["Demand"] for r in ROWS)
q1 = demands[len(demands) // 3]            # low/mid cut
q2 = demands[(2 * len(demands)) // 3]      # mid/high cut


def demand_bin(d: int) -> int:
    # Map a raw demand number → bin label 0, 1, or 2
    if d <= q1:
        return 0
    if d <= q2:
        return 1
    return 2


# Count bins, then convert to probabilities that sum to 1
_counts = Counter(demand_bin(r["Demand"]) for r in ROWS)
DEMAND_P = {k: _counts[k] / N_DAYS for k in (0, 1, 2)}

print("Demand bins P(d):", {k: round(v, 3) for k, v in DEMAND_P.items()}, "| cuts", q1, q2)

if plt is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    # Left: demand-bin probabilities
    axes[0].bar(["0", "1", "2"], [DEMAND_P[0], DEMAND_P[1], DEMAND_P[2]], color="#2F5D50")
    axes[0].set_title("Real Electronics demand bins")
    axes[0].set_xlabel("d")
    axes[0].set_ylabel("P(d)")
    axes[0].set_ylim(0, 1)
    # Right: promo vs not
    axes[1].bar(
        ["promo", "no promo"],
        [len(PROMO_DAYS), N_DAYS - len(PROMO_DAYS)],
        color=["#C45C26", "#2F5D50"],
    )
    axes[1].set_title("Real promotional days")
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib not available — numbers above are enough")


### 1b — TRY THIS (read the printout)

Fill three facts from Stop 1a. No inventing numbers.


In [ ]:
# STOP 1b — TRY THIS

n_days_guess = None        # <<< FILL: Days in log (integer)
promo_count_guess = None   # <<< FILL: Promo days (integer)
has_profit_column = None   # <<< FILL: True/False — does the log include Profit?

print("Your answers:", n_days_guess, promo_count_guess, has_profit_column)

if None in (n_days_guess, promo_count_guess, has_profit_column):
    print("NOT YET — fill all three from the Stop 1a printout")
else:
    ok = (
        n_days_guess == N_DAYS
        and promo_count_guess == len(PROMO_DAYS)
        and has_profit_column is True
    )
    print("CHECK 1 —", "PASS" if ok else "NOT YET — compare to Days in log / Promo days")


## Stop 2 — Map real columns → state and action

![MDP mapping](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/mdp-inventory.png)

| Data `Action_Taken` | Code $a$ |
| --- | --- |
| Reduce | 0 |
| Maintain | 1 |
| Order | 2 |

State rule vs average demand:
- **Low** if inventory $< 0.75 \times$ avg demand  
- **Medium** if $< 1.5 \times$ avg demand  
- **High** otherwise  

Basestock practice policy: Low→Order(2), Medium→Maintain(1), High→Reduce(0).


### 2a — Define `observe_state` (RUN WITH ME)


In [ ]:
# STOP 2a-1 — RUN WITH ME · observe_state

def observe_state(inventory, avg_demand=AVG_DEMAND):
    # Convert a raw inventory number into Low / Medium / High
    inv = float(inventory)
    if inv < 0.75 * avg_demand:
        return "Low"
    if inv < 1.5 * avg_demand:
        return "Medium"
    return "High"


print("Demo observe_state(0) →", observe_state(0))
print("Demo observe_state(avg) →", observe_state(AVG_DEMAND))
print("Demo observe_state(3*avg) →", observe_state(3 * AVG_DEMAND))


### 2a — Define action helpers (RUN WITH ME)


In [ ]:
# STOP 2a-2 — RUN WITH ME · action_to_code + decide_basestock

def action_to_code(action_taken: str) -> int:
    # Map historical Action_Taken text → {0,1,2}
    mapping = {"Reduce": 0, "Maintain": 1, "Order": 2}
    return mapping.get(action_taken, -1)  # -1 means unexpected label


def decide_basestock(state: str) -> int:
    # Simple class policy we practice tonight
    return {"Low": 2, "Medium": 1, "High": 0}[state]


print("action_to_code('Order') →", action_to_code("Order"))
print("decide_basestock('Low') →", decide_basestock("Low"))


### 2a — Show five real days (RUN WITH ME)


In [ ]:
# STOP 2a-3 — RUN WITH ME · peek at real rows

print("avg demand =", round(AVG_DEMAND, 1))
for r in ROWS[0:5]:
    s = observe_state(r["Current_Inventory"])  # our state label
    hist = r["Action_Taken"]                   # what the log did
    print(
        r["Date"],
        "inv=", r["Current_Inventory"],
        "→", s,
        "| data=", hist, action_to_code(hist),
        "| basestock=", decide_basestock(s),
    )


### 2b — TRY THIS — classify three inventories


In [ ]:
# STOP 2b — TRY THIS

inv_A = ROWS[0]["Current_Inventory"]  # given from the real log
inv_B = int(AVG_DEMAND)               # near average
inv_C = int(2.5 * AVG_DEMAND)         # clearly high

state_A = ""  # <<< FILL: "Low" / "Medium" / "High"
state_B = ""  # <<< FILL
state_C = ""  # <<< FILL
a_A = None    # <<< FILL: 0 / 1 / 2 from decide_basestock
a_B = None    # <<< FILL
a_C = None    # <<< FILL

print(inv_A, state_A, a_A)
print(inv_B, state_B, a_B)
print(inv_C, state_C, a_C)

if not state_A or not state_B or not state_C or None in (a_A, a_B, a_C):
    print("NOT YET — fill states and actions")
else:
    ok = (
        (state_A, a_A) == (observe_state(inv_A), decide_basestock(observe_state(inv_A)))
        and (state_B, a_B) == (observe_state(inv_B), decide_basestock(observe_state(inv_B)))
        and (state_C, a_C) == (observe_state(inv_C), decide_basestock(observe_state(inv_C)))
    )
    print("CHECK 2 —", "PASS" if ok else "NOT YET — use observe_state / decide_basestock")


### 2c — CHANGE ME · discuss thresholds together

**PAIR 20s.** Change the cutoffs. Re-run. Watch how many days become Low / Medium / High.

**Discuss:** If almost everything is “Low,” your basestock will Order too often. What does that do to holding cost?


In [ ]:
# STOP 2c — CHANGE ME · then discuss

# <<< CHANGE ME: try (0.5, 1.2) or (1.0, 2.0) and compare
LOW_CUT = 0.75    # inventory < LOW_CUT * avg demand → Low
HIGH_CUT = 1.5    # inventory < HIGH_CUT * avg demand → Medium; else High


def observe_state_custom(inventory, avg_demand=AVG_DEMAND, low_cut=LOW_CUT, high_cut=HIGH_CUT):
    # Same idea as observe_state, but YOU control the cuts
    inv = float(inventory)
    if inv < low_cut * avg_demand:
        return "Low"
    if inv < high_cut * avg_demand:
        return "Medium"
    return "High"


# Count how the whole real log splits under YOUR cuts
counts = {"Low": 0, "Medium": 0, "High": 0}
for r in ROWS:
    counts[observe_state_custom(r["Current_Inventory"])] += 1

print("Cuts: Low <", LOW_CUT, "* avg | Medium <", HIGH_CUT, "* avg")
print("Day counts:", counts)
print("Share Low:", round(counts["Low"] / N_DAYS, 3))
print("Discuss: does this match how aggressive you want ordering to be?")


### 2d — CHANGE ME · rewrite the policy and compare to history

**PAIR 20s.** Change what Low/Medium/High should do. Re-run.

**Discuss:** On how many real days would YOUR policy match the log’s `Action_Taken`?


In [ ]:
# STOP 2d — CHANGE ME · policy vs real history

# <<< CHANGE ME: map each state → 0 Reduce / 1 Maintain / 2 Order
POLICY = {
    "Low": 2,      # default: Order when low
    "Medium": 1,   # default: Maintain
    "High": 0,     # default: Reduce
}


def decide_custom(state: str) -> int:
    return POLICY[state]


# Compare YOUR policy to the historical Action_Taken on every real day
agree = 0
for r in ROWS:
    s = observe_state(r["Current_Inventory"])          # use original classroom cuts
    mine = decide_custom(s)                            # your policy
    hist = action_to_code(r["Action_Taken"])           # what the log did
    if mine == hist:
        agree += 1

print("Your POLICY:", POLICY)
print("Agree with historical Action_Taken on", agree, "/", N_DAYS, "days (", round(100 * agree / N_DAYS, 1), "%)")
print("Discuss: matching history is NOT required — but big disagreement needs a story.")


## Stop 3 — Solve one REAL day

![Real day](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-agent-env.png)
![Promo path](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/llm-tool-loop.png)

We pick a real non-promo day and a real promo day from `ROWS`, then build a promo brief from that date's numbers.


### 3a — Helper to pick a real day (RUN WITH ME)


In [ ]:
# STOP 3a-1 — RUN WITH ME · pick_real_day

def pick_real_day(prefer_promo: bool):
    # Walk the log and return the first day matching promo / not-promo
    for r in ROWS:
        if r["Is_Promotional"] is prefer_promo:
            return r
    # Fallback (should not happen with this dataset)
    return ROWS[0]


print("Helper ready: pick_real_day(True/False)")


### 3a — Load two real days (RUN WITH ME)


In [ ]:
# STOP 3a-2 — RUN WITH ME · DAY_NORMAL and DAY_PROMO

DAY_NORMAL = pick_real_day(False)  # Is_Promotional == False
DAY_PROMO = pick_real_day(True)    # Is_Promotional == True

# Columns we care about for class discussion
keys = [
    "Date",
    "Current_Inventory",
    "Demand",
    "Stockout_Units",
    "Is_Promotional",
    "Action_Taken",
    "Order_Quantity",
    "Profit",
    "Holding_Cost",
    "Stockout_Cost",
]

print("REAL DAY — normal")
print({k: DAY_NORMAL[k] for k in keys})
print(
    " state=", observe_state(DAY_NORMAL["Current_Inventory"]),
    " basestock=", decide_basestock(observe_state(DAY_NORMAL["Current_Inventory"])),
)

print()
print("REAL DAY — promotional")
print({k: DAY_PROMO[k] for k in keys})
print(
    " state=", observe_state(DAY_PROMO["Current_Inventory"]),
    " basestock=", decide_basestock(observe_state(DAY_PROMO["Current_Inventory"])),
)


### 3a — Build the exception document (RUN WITH ME)

The brief uses **real numbers** from `DAY_PROMO`, but it is still a **document** (not a stock column).


In [ ]:
# STOP 3a-3 — RUN WITH ME · PROMO_BRIEF

PROMO_BRIEF = {
    "date": DAY_PROMO["Date"],
    "subject": "Electronics campaign live — " + DAY_PROMO["Date"],
    "body": (
        "Marketing confirms promotional day. demand="
        + str(DAY_PROMO["Demand"])
        + ", inventory="
        + str(DAY_PROMO["Current_Inventory"])
        + ", stockout_units="
        + str(DAY_PROMO["Stockout_Units"])
        + ". Do not auto-change Order_Quantity without desk-lead stamp."
    ),
}

print("EXCEPTION DOCUMENT:")
print(" ", PROMO_BRIEF["subject"])
print(" ", PROMO_BRIEF["body"])


### 3b — TRY THIS — promo day questions


In [ ]:
# STOP 3b — TRY THIS

promo_date = ""  # <<< FILL: DAY_PROMO["Date"]
base_a = None    # <<< FILL: basestock action 0/1/2 for that day's inventory
needs_override_draft = None  # <<< FILL: True/False
email_inside_inventory_number = None  # <<< FILL: True/False

print(promo_date, base_a, needs_override_draft, email_inside_inventory_number)

if not promo_date or base_a is None or needs_override_draft is None or email_inside_inventory_number is None:
    print("NOT YET")
else:
    ok = (
        promo_date == DAY_PROMO["Date"]
        and base_a == decide_basestock(observe_state(DAY_PROMO["Current_Inventory"]))
        and needs_override_draft is True
        and email_inside_inventory_number is False
    )
    print("CHECK 3 —", "PASS" if ok else "NOT YET")


### 3c — CHANGE ME · pick another real day and discuss

**PAIR 20s.** Set `DAY_INDEX` to another row (0 … N_DAYS-1). Re-run.

**Discuss:** Is this day promotional? Would you draft an override? Who stamps it?


In [ ]:
# STOP 3c — CHANGE ME · explore another real day

DAY_INDEX = 100  # <<< CHANGE ME: try 0, 50, 200, 400, ...

day = ROWS[DAY_INDEX]  # one real dictionary from the log
state = observe_state(day["Current_Inventory"])
base_a = decide_basestock(state)

print("Date:", day["Date"])
print("Inventory:", day["Current_Inventory"], "→", state, "| basestock a =", base_a)
print("Demand:", day["Demand"], "| Stockout:", day["Stockout_Units"])
print("Is_Promotional:", day["Is_Promotional"])
print("Historical Action_Taken:", day["Action_Taken"], "| Order_Quantity:", day["Order_Quantity"])
print("Profit $", day["Profit"], "| Holding $", day["Holding_Cost"], "| Stockout cost $", day["Stockout_Cost"])

# Your quick call for discussion (not graded by assert)
would_draft_override = day["Is_Promotional"]  # <<< CHANGE ME to True/False if you disagree
print("Would draft override?", would_draft_override)
print("Discuss: proposal still needs a human stamp before release.")


## Stop 4 — Choose the family (TRY THIS)

![Choose family](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/two-families.png)

Type exactly: `LLM`, `RL`, `hybrid`, or `do not automate`.


In [ ]:
# STOP 4 — TRY THIS

# A) Only Current_Inventory → Order/Maintain/Reduce daily; profit costs; no briefs
case_A = ""  # <<< FILL

# B) One campaign brief; human stamp; not the daily controller alone
case_B = ""  # <<< FILL

# C) THIS desk: 730 inventory days AND many Is_Promotional days
case_C = ""  # <<< FILL

# D) No signed reward policy; autonomous orders forbidden for now
case_D = ""  # <<< FILL

print(case_A, case_B, case_C, case_D)

if any(not x.strip() for x in [case_A, case_B, case_C, case_D]):
    print("NOT YET — fill A–D")
else:
    ok = [case_A, case_B, case_C, case_D] == ["RL", "LLM", "hybrid", "do not automate"]
    print("CHECK 4 —", "PASS" if ok else "NOT YET — expected RL / LLM / hybrid / do not automate")


## Stop 5 — DEFINE the MDP side (TRY THIS)

![MDP](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-mdp.png)

Use **real column names** (`Current_Inventory`, `Profit`, …).


In [ ]:
# STOP 5 — TRY THIS

mdp_states = ""          # <<< FILL: mention Current_Inventory
mdp_actions = []         # <<< FILL: [0, 1, 2]
mdp_reward = ""          # <<< FILL: must mention profit OR sales OR margin
mdp_why_repeatable = ""  # <<< FILL: daily / Electronics / N_DAYS

print("states:", mdp_states or "(FILL)")
print("actions:", mdp_actions or "(FILL)")
print("reward:", mdp_reward or "(FILL)")
print("why:", mdp_why_repeatable or "(FILL)")
print("Evidence N_DAYS =", N_DAYS)

if not mdp_states or not mdp_actions or not mdp_reward or not mdp_why_repeatable:
    print("NOT YET")
else:
    ok = (
        mdp_actions == [0, 1, 2]
        and any(w in mdp_reward.lower() for w in ("margin", "sales", "profit"))
        and "inventory" in mdp_states.lower()
    )
    print("CHECK 5 —", "PASS" if ok else "NOT YET")


## Stop 6 — DEFINE the LLM side on the real promo brief

![LLM](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-llm-agent.png)
![Stamp](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-handoff.png)


### 6a — Re-read the real brief (RUN WITH ME)


In [ ]:
# STOP 6a — RUN WITH ME

print(PROMO_BRIEF["subject"])
print(PROMO_BRIEF["body"])

# What basestock would do on that inventory (numeric path)
base = decide_basestock(observe_state(DAY_PROMO["Current_Inventory"]))
print("Basestock a =", base)

# What the historical log actually did that day
print(
    "Historical Action_Taken =",
    DAY_PROMO["Action_Taken"],
    "| Order_Quantity =",
    DAY_PROMO["Order_Quantity"],
)


### 6b — TRY THIS — LLM design + one draft override

Remember: **proposal ≠ release**.


In [ ]:
# STOP 6b — TRY THIS

llm_goal = ""            # <<< FILL
llm_tools = []           # <<< FILL: list of tool name strings
llm_knowledge = ""       # <<< FILL
llm_human_handoff = ""   # <<< FILL: who stamps?
proposed_override_a = None  # <<< FILL: 0, 1, or 2
proposal_reason = ""        # <<< FILL: must include the real promo DATE

print("goal:", llm_goal or "(FILL)")
print("tools:", llm_tools or "(FILL)")
print("knowledge:", llm_knowledge or "(FILL)")
print("handoff:", llm_human_handoff or "(FILL)")
print("proposal:", proposed_override_a, "|", proposal_reason or "(FILL)")

if (
    not llm_goal
    or not llm_tools
    or not llm_human_handoff
    or proposed_override_a is None
    or not proposal_reason
):
    print("NOT YET")
else:
    ok = proposed_override_a in (0, 1, 2) and DAY_PROMO["Date"] in proposal_reason
    print("CHECK 6 —", "PASS" if ok else "NOT YET — include the real DATE in proposal_reason")


### 6c — CHANGE ME · draft a different override and discuss

**PAIR 20s.** Change `proposed_override_a`. Re-run.

**Discuss:** On the real promo day, when is Order (2) vs Maintain (1) the better draft — and who can release it?


In [ ]:
# STOP 6c — CHANGE ME · override draft options

print("Promo date:", DAY_PROMO["Date"])
print("Inventory:", DAY_PROMO["Current_Inventory"], "→", observe_state(DAY_PROMO["Current_Inventory"]))
print("Basestock would do a =", decide_basestock(observe_state(DAY_PROMO["Current_Inventory"])))
print("Historical Action_Taken:", DAY_PROMO["Action_Taken"])

proposed_override_a = 2  # <<< CHANGE ME: try 0, 1, or 2
print("Your draft override a =", proposed_override_a)

if proposed_override_a == decide_basestock(observe_state(DAY_PROMO["Current_Inventory"])):
    print("Draft matches basestock — then why open an LLM path today?")
else:
    print("Draft DIFFERS from basestock — say out loud what risk the promo brief adds.")

print("Discuss: the LLM may propose", proposed_override_a, "but desk lead still stamps before release.")


## Stop 7 — Memo for THIS real Electronics desk

![Memo](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/hybrid.png)

We split this stop: vote → release fields → define `review_memo` → assemble `MEMO` → run check.


### 7a — Class vote (TRY THIS)


In [ ]:
# STOP 7a — TRY THIS

say_it = ""  # <<< FILL: LLM / RL / hybrid / do not automate
print("Vote:", say_it or "(FILL)")
print("Evidence — promo days:", len(PROMO_DAYS), "/", N_DAYS)
print("Hint: Stop 4 case C for THIS desk is usually hybrid.")


### 7b — Who releases what + because (TRY THIS)


In [ ]:
# STOP 7b — TRY THIS

releases_daily_order = ""     # <<< FILL: who releases the daily order?
releases_email_override = ""  # <<< FILL: who stamps the promo override?
because_text = ""             # <<< FILL: ≥ 40 chars; mention Electronics or inventory_dataset_full

print("Order release:", releases_daily_order or "(FILL)")
print("Override release:", releases_email_override or "(FILL)")
print("Because:", because_text or "(FILL)")


### 7c — Define `reward_ok` (RUN WITH ME)

Tiny helper: does the reward sentence use CFO language?


In [ ]:
# STOP 7c-1 — RUN WITH ME · reward_ok

def reward_ok(text: str) -> bool:
    # Accept margin OR sales OR profit (Session 5 / CFO language)
    lowered = (text or "").lower()
    return "margin" in lowered or "sales" in lowered or "profit" in lowered


print("reward_ok('Profit minus holding') →", reward_ok("Profit minus holding"))
print("reward_ok('be nice') →", reward_ok("be nice"))


### 7c — Define `review_memo` (RUN WITH ME)

Read each `issues.append` — that is the grading checklist.


In [ ]:
# STOP 7c-2 — RUN WITH ME · review_memo

def review_memo(memo: dict) -> bool:
    # Score the go/no-go memo. Print PASS or a short fix list.
    issues = []
    llm = memo.get("llm_design", {})
    mdp = memo.get("mdp_design", {})
    rec = (memo.get("recommendation") or "").strip()
    because = (memo.get("because") or "").strip()
    who_order = (memo.get("releases_daily_order") or "").strip()
    who_email = (memo.get("releases_email_override") or "").strip()

    # LLM side complete?
    if not (llm.get("goal") or "").strip() or not llm.get("tools") or not (llm.get("human_handoff") or "").strip():
        issues.append("LLM needs goal, tools, human_handoff.")

    # MDP side complete?
    if not (mdp.get("states") or "").strip() or not mdp.get("actions") or not reward_ok(mdp.get("reward", "")):
        issues.append("MDP needs states, actions, reward with margin/sales/profit.")
    if not (mdp.get("why_repeatable") or "").strip():
        issues.append("MDP needs why_repeatable.")

    # Recommendation legal?
    if rec not in {"LLM", "RL", "hybrid", "do not automate"}:
        issues.append("recommendation must be LLM / RL / hybrid / do not automate.")

    # Wrong family trap: numeric inventory + LLM-only
    if rec == "LLM" and "inventory" in (mdp.get("states") or "").lower():
        issues.append("inventory state + LLM-only → usually hybrid")

    # Because must be readable and cite this real case
    if len(because) < 40:
        issues.append("because too short (≥ 40 characters)")
    if "electronics" not in because.lower() and "inventory_dataset" not in because.lower():
        issues.append("because must mention Electronics or inventory_dataset_full")

    # Accountability
    if len(who_order) < 3 or len(who_email) < 3:
        issues.append("name who releases daily order AND who releases promo override")

    if issues:
        print("NOT YET")
        for issue in issues:
            print(" -", issue)
        return False

    print("PASS — real-case memo ready to read aloud")
    print("Recommendation:", rec)
    print("Because:", because)
    print("Order release:", who_order)
    print("Override release:", who_email)
    print("Evidence days:", N_DAYS, "| promo days:", len(PROMO_DAYS), "| promo date:", DAY_PROMO["Date"])
    return True


print("Defined review_memo(memo) → True/False")


### 7c — Assemble `MEMO` and run the check (RUN / TRY)

Uses your Stop 5–7b variables. Re-run earlier FILLs if something is empty.


In [ ]:
# STOP 7c-3 — assemble MEMO + review

MEMO = {
    "process": "Electronics replenishment exceptions (inventory_dataset_full.csv)",
    "llm_design": {
        "goal": llm_goal,
        "tools": list(llm_tools),
        "knowledge": llm_knowledge,
        "human_handoff": llm_human_handoff,
    },
    "mdp_design": {
        "states": mdp_states,
        "actions": list(mdp_actions),
        "reward": mdp_reward,
        "why_repeatable": mdp_why_repeatable,
    },
    "recommendation": (say_it or "").strip(),
    "releases_daily_order": releases_daily_order,
    "releases_email_override": releases_email_override,
    "because": because_text,
}

print("Assembled recommendation:", MEMO["recommendation"] or "(missing Stop 7a)")
review_memo(MEMO)


## Stop 8 — LAB (10–15 min) on real rows

![Keep the loop](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/agent-loop.png)


### Challenge 1 — Mean profit: promo vs normal


In [ ]:
# CH1 — TRY THIS

promo_profit = None   # <<< FILL: mean Profit where Is_Promotional is True
normal_profit = None  # <<< FILL: mean Profit where Is_Promotional is False

# Hint:
# promo_profit = sum(r["Profit"] for r in ROWS if r["Is_Promotional"]) / len(PROMO_DAYS)
# normal_profit = sum(r["Profit"] for r in ROWS if not r["Is_Promotional"]) / (N_DAYS - len(PROMO_DAYS))

print(promo_profit, normal_profit)
if promo_profit is None or normal_profit is None:
    print("NOT YET — compute from ROWS")
else:
    print("CH1 recorded. Discuss: does promo change money enough to need an override path?")


### Challenge 1b — CHANGE ME · filter the real log

**PAIR 20s.** Change `ONLY_PROMO`. Re-run mean profit.

**Discuss:** Do promo days look more profitable here? Does that alone justify auto-ordering?


In [ ]:
# CH1b — CHANGE ME · discuss money

ONLY_PROMO = True  # <<< CHANGE ME: True = promo days only; False = non-promo only

subset = [r for r in ROWS if r["Is_Promotional"] is ONLY_PROMO]
mean_profit = sum(r["Profit"] for r in subset) / max(1, len(subset))
mean_stockout_cost = sum(r["Stockout_Cost"] for r in subset) / max(1, len(subset))

print("ONLY_PROMO =", ONLY_PROMO, "| days =", len(subset))
print("Mean Profit $", round(mean_profit, 1))
print("Mean Stockout_Cost $", round(mean_stockout_cost, 1))
print("Discuss: money changed — who is allowed to change Order_Quantity because of that?")


### Challenge 2 — Worst stockout day in the real log


In [ ]:
# CH2 — TRY THIS

day = max(ROWS, key=lambda r: r["Stockout_Units"])  # real worst stockout day
print("Worst stockout:", day["Date"], day["Stockout_Units"], "promo=", day["Is_Promotional"])

family_for_recurring_stockout_plus_notes = ""  # <<< FILL if ops notes accompany stockouts
print("CHECK:", "PASS" if family_for_recurring_stockout_plus_notes == "hybrid" else "NOT YET — usually hybrid")


### Challenge 3 — Break the memo on purpose, then fix (RUN WITH ME)


In [ ]:
# CH3 — RUN WITH ME

# Copy MEMO and deliberately choose the wrong family
bad = dict(MEMO)
bad["recommendation"] = "LLM"  # wrong for a numeric inventory desk

print("Broken on purpose:")
review_memo(bad)

print()
print("Fixed (your real MEMO):")
review_memo(MEMO)


## Stop 9 — Framework line (keep forever)

One sentence for **your** project: observe / decide / reward / handoff.


In [ ]:
# STOP 9 — TRY THIS

framework_family = ""  # <<< FILL: LLM / RL / hybrid
framework_line = ""    # <<< FILL: one keepable sentence

print("Family:", framework_family or "(FILL)")
print("I keep:", framework_line or "(FILL)")


## Raise a hand — real case complete

1. [ ] I can explain each Stop 1 helper (`course_data`, `load_electronics_desk`, stats)  
2. [ ] I mapped real inventory → state/action  
3. [ ] I solved a **real promo date** (base $a$ + override draft)  
4. [ ] `review_memo` prints **PASS**  
5. [ ] I can say who releases the daily order vs the promo override  

Session 7 fills **decide** with values (Bellman / Q) on the same numeric path.
